# Joining BasinAtlas data to HUC8 watersheds

In [2]:
import geopandas as gpd
import pandas as pd
from config import *

In [3]:
huc8 = gpd.read_file(geospat_data_filepath+'huc8_shapefiles/HUC8_US_5070.shp')

# Read in basin atlas data
basinatlas = gpd.read_file(INPUT_filepath+'04_static_basinatlas_catchment_characteristics/BasinATLAS_v10_shp/BasinATLAS_v10_lev12.shp')

# clip basin atlas shapefile to just the US
basinatlas_NA = gpd.clip(basinatlas, (-172, 20, -67, 50), keep_geom_type=True)
basinatlas_NA = basinatlas_NA.to_crs(huc8.crs)

In [4]:
# Intersecting the watersheds
basin_intersection = gpd.overlay(basinatlas_NA, 
                                 huc8[['HUC8','AREASQKM','geometry']],
                                 how='intersection')

In [5]:
# Calculating the area of the basin atlas watersheds within the HUC8 watershed. 
basin_intersection['partial_area'] = basin_intersection.area / 1e6  # km²

basin_intersection['total_intersected_area'] = (
    basin_intersection.groupby('HUC8')['partial_area'].transform('sum')
)

basin_intersection['weight'] = (
    basin_intersection['partial_area'] / basin_intersection['total_intersected_area']
)
basin_intersection.tail()

,HYBAS_ID,NEXT_DOWN,NEXT_SINK,MAIN_BAS,DIST_SINK,DIST_MAIN,SUB_AREA,UP_AREA,PFAF_ID,ENDO,...,gdp_ud_sav,gdp_ud_ssu,gdp_ud_usu,hdi_ix_sav,HUC8,AREASQKM,geometry,partial_area,total_intersected_area,weight
121426,7120335000,7120340710,7120038320,7120038320,507.3,507.3,207.1,207.2,726087460000,0,...,37544,24123362,24123360,916,01010008,6027.12,"POLYGON ((2011481.086 3065265.263, 2011620.009...",202.488008,6027.115758,0.033596
121427,7120982590,7120331560,7120034710,7120034710,26.0,26.0,159.0,159.0,726013062200,0,...,37544,13131531,13131531,916,01010008,6027.12,"MULTIPOLYGON (((2017318.757 3071529.932, 20157...",2.057215,6027.115758,0.000341
121428,7120327310,7120325560,7120034770,7120034770,41.4,41.4,123.5,123.5,726013086000,0,...,37544,5974531,5974532,916,01010008,6027.12,"MULTIPOLYGON (((2029887.265 3076915.484, 20288...",6.711529,6027.115758,0.001114
121429,7120325670,7120323560,7120034770,7120034770,23.9,23.9,155.1,155.1,726013084000,0,...,37544,2889836,2889836,916,01010008,6027.12,"MULTIPOLYGON (((2027228.358 3079607.938, 20272...",0.950306,6027.115758,0.000158
121430,7120980880,7120327210,7120034770,7120034770,52.1,52.1,150.5,620.2,726013087200,0,...,37544,8647379,8849472,916,01010008,6027.12,"MULTIPOLYGON (((2050632.695 3085734.46, 205027...",1.396976,6027.115758,0.000232


In [6]:
# adjusting the variables so they are in their proper (unscaled) units
basin_intersection['ari_ix_sav'] = basin_intersection['ari_ix_sav']/100
basin_intersection['slp_dg_sav'] = basin_intersection['slp_dg_sav']/10

In [7]:
# Use weights to scale the paramteres below
upstream_cols = ['ari_ix_sav', 'kar_pc_sse', 'ppd_pk_sav', 'rdd_mk_sav',
                 'hft_ix_s09', 'slp_dg_sav', 'ele_mt_sav']
majority_cols = ['clz_cl_smj', 'lit_cl_smj']

# Scaling the upstream values based on weights
us_scaled = basin_intersection[upstream_cols].mul(basin_intersection['weight'], axis=0)
us_scaled = pd.concat([basin_intersection['HUC8'], us_scaled], axis=1)
upstream_properties = us_scaled.groupby('HUC8').sum().reset_index()
upstream_properties.head()

,HUC8,ari_ix_sav,kar_pc_sse,ppd_pk_sav,rdd_mk_sav,hft_ix_s09,slp_dg_sav,ele_mt_sav
0,01010002,1.317808,0.000000,0.022995,0.330676,3.376635,3.696777,364.239887
1,01010003,1.288682,0.012406,3.398320,59.319011,30.739143,4.195638,276.806660
2,01010004,1.234440,26.746064,5.603931,109.398907,46.196235,2.811273,252.189266
3,01010005,1.216960,23.672699,13.004249,236.298967,76.588756,2.227957,177.873176
4,01010006,1.379090,0.068308,2.164368,45.532089,25.937115,2.218876,428.012033


In [8]:
maj = ['clz_cl_smj', 'lit_cl_smj']

# taking the value of the variable's (clim and lith, seperately) class that has the highest coverage in the HUC8 watersehd. 
long = basin_intersection.melt(
    id_vars=['HUC8', 'weight'],
    value_vars=maj,
    var_name='variable', value_name='class'
).dropna(subset=['class'])

dominant_properties = (
    long.groupby(['HUC8', 'variable', 'class'])['weight'].sum()
    .reset_index(name='class_weight')
    .sort_values(['HUC8', 'variable', 'class_weight', 'class'],
                 ascending=[True, True, False, True])
    .groupby(['HUC8', 'variable']).head(1)
    .pivot(index='HUC8', columns='variable', values='class')
    .reset_index()
)
HUC8_properties = upstream_properties.merge(dominant_properties, on = 'HUC8')
HUC8_properties.head()

,HUC8,ari_ix_sav,kar_pc_sse,ppd_pk_sav,rdd_mk_sav,hft_ix_s09,slp_dg_sav,ele_mt_sav,clz_cl_smj,lit_cl_smj
0,01010002,1.317808,0.000000,0.022995,0.330676,3.376635,3.696777,364.239887,7,3
1,01010003,1.288682,0.012406,3.398320,59.319011,30.739143,4.195638,276.806660,7,3
2,01010004,1.234440,26.746064,5.603931,109.398907,46.196235,2.811273,252.189266,7,3
3,01010005,1.216960,23.672699,13.004249,236.298967,76.588756,2.227957,177.873176,7,6
4,01010006,1.379090,0.068308,2.164368,45.532089,25.937115,2.218876,428.012033,7,3


In [10]:
HUC8_properties.dtypes

HUC8              str
ari_ix_sav    float64
kar_pc_sse    float64
ppd_pk_sav    float64
rdd_mk_sav    float64
hft_ix_s09    float64
slp_dg_sav    float64
ele_mt_sav    float64
clz_cl_smj      int32
lit_cl_smj      int32
dtype: object

In [12]:
HUC8_properties.to_csv(INPUT_filepath+'04_static_basinatlas_catchment_characteristics/huc8_basinatlas_properties.csv', index=False)